<a href="https://colab.research.google.com/github/zuizui0223/mc/blob/main/0416.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import requests
import json

# iNaturalist APIのエンドポイント
API_URL = "https://api.inaturalist.org/v1/observations"

# 検索クエリのパラメータを設定
# 訪花昆虫（ハチ、チョウ、ハエ）のタクソンIDを設定
# ハチ (Apidae): 47217, チョウ (Rhopalocera): 47224, ハエ (Diptera): 47192
base_params = {
    "taxon_id": "47157", # 訪花昆虫のタクソンIDを複数指定
    "photos": "true",    # 写真があるもののみ
    "identified": "true", # 識別済みのもののみ
    "per_page": 200,     # 1ページあたりの結果数（最大200に増加）
    "place_id": "6737"   # 日本に限定（ユーザーからの情報に基づいて6737に修正）
}

all_results = []
total_results = 0
num_pages_to_fetch = 10 # ここで取得するページ数を指定します

print(f"iNaturalist APIから {num_pages_to_fetch} ページ分のデータを取得します... (訪花昆虫、日本に限定)")

for page_num in range(1, num_pages_to_fetch + 1):
    params = base_params.copy()
    params["page"] = page_num

    print(f"  ページ {page_num}/{num_pages_to_fetch} をリクエスト中... {API_URL}?{requests.compat.urlencode(params)}")
    response = requests.get(API_URL, params=params)

    if response.status_code == 200:
        page_data = response.json()
        if page_num == 1: # 最初のページで総件数を取得
            total_results = page_data['total_results']
        all_results.extend(page_data['results'])
        print(f"  ページ {page_num} で {len(page_data['results'])} 件の観察データを取得しました。")
        if not page_data['results']: # 取得できるデータがなくなったらループを抜ける
            break
    else:
        print(f"エラーが発生しました: {response.status_code} - {response.text}")
        break

# 全てのデータを格納する辞書を作成
data = {
    'total_results': total_results,
    'page': num_pages_to_fetch, # 最終的に取得したページ数（またはリクエストしたページ数）
    'per_page': base_params['per_page'],
    'results': all_results
}

print(f"\n合計 {data['total_results']} 件中、{len(data['results'])} 件の観察データが見つかりました（{num_pages_to_fetch} ページから）。")
print("最初の5件の観察データのタイトルと写真URLを表示します:")
for i, result in enumerate(data['results'][:5]):
    description = result.get('description', 'No description')
    print(f"--- Observation {i+1} ---")
    print(f"Species: {result['species_guess']}")
    print(f"Observed on: {result['observed_on']}")
    # 最初の写真のURLを取得
    if result['photos']:
        print(f"Photo URL: {result['photos'][0]['url']}")
    else:
        print("No photo available for this observation.")

iNaturalist APIから 10 ページ分のデータを取得します... (訪花昆虫、日本に限定)
  ページ 1/10 をリクエスト中... https://api.inaturalist.org/v1/observations?taxon_id=47157&photos=true&identified=true&per_page=200&place_id=6737&page=1
  ページ 1 で 200 件の観察データを取得しました。
  ページ 2/10 をリクエスト中... https://api.inaturalist.org/v1/observations?taxon_id=47157&photos=true&identified=true&per_page=200&place_id=6737&page=2
  ページ 2 で 200 件の観察データを取得しました。
  ページ 3/10 をリクエスト中... https://api.inaturalist.org/v1/observations?taxon_id=47157&photos=true&identified=true&per_page=200&place_id=6737&page=3
  ページ 3 で 200 件の観察データを取得しました。
  ページ 4/10 をリクエスト中... https://api.inaturalist.org/v1/observations?taxon_id=47157&photos=true&identified=true&per_page=200&place_id=6737&page=4
  ページ 4 で 200 件の観察データを取得しました。
  ページ 5/10 をリクエスト中... https://api.inaturalist.org/v1/observations?taxon_id=47157&photos=true&identified=true&per_page=200&place_id=6737&page=5
  ページ 5 で 200 件の観察データを取得しました。
  ページ 6/10 をリクエスト中... https://api.inaturalist.org/v1/observations?taxon_id=47157&ph

In [6]:
import os
import requests
from PIL import Image
from io import BytesIO

# ダウンロードする画像の保存先ディレクトリ
DOWNLOAD_DIR = "inaturalist_images"
if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)

# 画像ダウンロード関数
def download_image(url, filename):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status() # HTTPエラーを確認
        img = Image.open(BytesIO(response.content))
        # PNG形式で保存 (JPGの場合もあるが、ここでは統一)
        img.save(os.path.join(DOWNLOAD_DIR, filename + ".png"))
        return True
    except requests.exceptions.RequestException as e:
        print(f"画像ダウンロードエラー {url}: {e}")
        return False
    except Exception as e:
        print(f"画像処理エラー {url}: {e}")
        return False

print(f"{data['total_results']}件の観察データから画像をダウンロードします（写真があるもののみ）。")

downloaded_count = 0
for i, result in enumerate(data['results']):
    if result['photos']:
        photo_url = result['photos'][0]['url']
        # ファイル名をユニークにするために観察IDを使用
        observation_id = result['id']
        filename = f"observation_{observation_id}"
        if download_image(photo_url, filename):
            downloaded_count += 1

        # 以前のダウンロード制限を解除しました。全ての画像がダウンロードされます。

print(f"{downloaded_count}枚の画像を {DOWNLOAD_DIR} にダウンロードしました。")

# ダウンロードしたファイルの確認 (最初の5ファイル)
print(f"\n{DOWNLOAD_DIR} ディレクトリのファイル一覧 (最初の5件):")
for f_name in os.listdir(DOWNLOAD_DIR)[:5]:
    print(f_name)

60564件の観察データから画像をダウンロードします（写真があるもののみ）。
2000枚の画像を inaturalist_images にダウンロードしました。

inaturalist_images ディレクトリのファイル一覧 (最初の5件):
observation_347785978.png
observation_345239171.png
observation_342382994.png
observation_341061650.png
observation_343283749.png


In [7]:
import numpy as np
from PIL import Image
import os

# ダウンロードディレクトリと画像サイズの定義
DOWNLOAD_DIR = "inaturalist_images"
IMAGE_SIZE = (128, 128) # モデル入力用に画像をリサイズするサイズ

# 観察IDと種別名のマッピングを作成
# data変数は前のセルでAPIから取得したレスポンス
observation_to_species = {}
for obs in data['results']:
    if obs['species_guess'] is not None and obs['species_guess'] != "None": # 'None'文字列も除外
        observation_to_species[str(obs['id'])] = obs['species_guess']

images = []
labels = []

# ダウンロードした画像を読み込み、前処理とラベル付けを行う
print("画像の読み込み、前処理（リサイズ、正規化）、ラベル抽出を開始します...")
for filename in os.listdir(DOWNLOAD_DIR):
    if filename.endswith(".png"):
        obs_id = filename.split('_')[1].split('.')[0]
        if obs_id in observation_to_species: # species_guessがある画像のみを対象
            img_path = os.path.join(DOWNLOAD_DIR, filename)
            try:
                img = Image.open(img_path).convert('RGB') # RGBに変換して3チャンネルにする
                img = img.resize(IMAGE_SIZE)             # 指定サイズにリサイズ
                img_array = np.array(img) / 255.0         # ピクセル値を0-1に正規化
                images.append(img_array)
                labels.append(observation_to_species[obs_id])
            except Exception as e:
                print(f"画像 {filename} の処理中にエラーが発生しました: {e}")

# リストをNumPy配列に変換
X_preprocessed = np.array(images)
y_species_preprocessed = np.array(labels)

print("\n画像の読み込み、リサイズ、正規化、およびラベルの抽出が完了しました。")
print(f"前処理後の総画像数: {len(X_preprocessed)}")
print(f"画像の形状: {X_preprocessed.shape[1:]}")
print(f"抽出されたユニークな種別数: {len(np.unique(y_species_preprocessed))}")

# 次のステップのために変数を準備
# これらは、後続のセルでラベルエンコーディングとデータ分割に使用されます。
# 現在は単に前処理された画像データとラベルの生データとして扱います。
# X は画像データ、y_species は対応する種別名（文字列）
X = X_preprocessed
y_species = y_species_preprocessed


画像の読み込み、前処理（リサイズ、正規化）、ラベル抽出を開始します...

画像の読み込み、リサイズ、正規化、およびラベルの抽出が完了しました。
前処理後の総画像数: 1855
画像の形状: (128, 128, 3)
抽出されたユニークな種別数: 423


In [8]:
import numpy as np
from PIL import Image
import os

# ダウンロードディレクトリと画像サイズの定義
DOWNLOAD_DIR = "inaturalist_images"
IMAGE_SIZE = (128, 128) # モデル入力用に画像をリサイズするサイズ

# 観察IDと種別名のマッピングを作成
# data変数は前のセルでAPIから取得したレスポンス
observation_to_species = {}
for obs in data['results']:
    if obs['species_guess'] is not None and obs['species_guess'] != "None": # 'None'文字列も除外
        observation_to_species[str(obs['id'])] = obs['species_guess']

images = []
labels = []

# ダウンロードした画像を読み込み、前処理とラベル付けを行う
print("画像の読み込み、前処理（リサイズ、正規化）、ラベル抽出を開始します...")
for filename in os.listdir(DOWNLOAD_DIR):
    if filename.endswith(".png"):
        obs_id = filename.split('_')[1].split('.')[0]
        if obs_id in observation_to_species: # species_guessがある画像のみを対象
            img_path = os.path.join(DOWNLOAD_DIR, filename)
            try:
                img = Image.open(img_path).convert('RGB') # RGBに変換して3チャンネルにする
                img = img.resize(IMAGE_SIZE)             # 指定サイズにリサイズ
                img_array = np.array(img) / 255.0         # ピクセル値を0-1に正規化
                images.append(img_array)
                labels.append(observation_to_species[obs_id])
            except Exception as e:
                print(f"画像 {filename} の処理中にエラーが発生しました: {e}")

# リストをNumPy配列に変換
X_preprocessed = np.array(images)
y_species_preprocessed = np.array(labels)

print("\n画像の読み込み、リサイズ、正規化、およびラベルの抽出が完了しました。")
print(f"前処理後の総画像数: {len(X_preprocessed)}")
print(f"画像の形状: {X_preprocessed.shape[1:]}")
print(f"抽出されたユニークな種別数: {len(np.unique(y_species_preprocessed))}")

# 次のステップのために変数を準備
# これらは、後続のセルでラベルエンコーディングとデータ分割に使用されます。
# 現在は単に前処理された画像データとラベルの生データとして扱います。
# X は画像データ、y_species は対応する種別名（文字列）
X = X_preprocessed
y_species = y_species_preprocessed


画像の読み込み、前処理（リサイズ、正規化）、ラベル抽出を開始します...

画像の読み込み、リサイズ、正規化、およびラベルの抽出が完了しました。
前処理後の総画像数: 1855
画像の形状: (128, 128, 3)
抽出されたユニークな種別数: 423


## 2. データセットの準備と前処理

ダウンロードした画像を機械学習モデルの入力として使用できるように準備します。ここでは、以下の処理を行います。

1.  **画像の読み込みとラベルの抽出**: ダウンロードした画像ファイルを読み込み、元のiNaturalist APIのデータ (`data`変数) から `species_guess` をラベルとして抽出します。`species_guess`が`None`のデータは除外します。
2.  **前処理**: モデルが効率的に学習できるように、すべての画像を同じサイズ（例: 128x128ピクセル）にリサイズし、ピクセル値を正規化（0-1の範囲に変換）します。
3.  **データ分割**: 訓練用とテスト用にデータセットを分割します。

In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from PIL import Image
import os

# ダウンロードディレクトリと画像サイズの定義
DOWNLOAD_DIR = "inaturalist_images"
IMAGE_SIZE = (128, 128) # モデル入力用に画像をリサイズするサイズ

# 観察IDと種別名のマッピングを作成
# data変数は前のセルでAPIから取得したレスポンス
observation_to_species = {}
for obs in data['results']:
    if obs['species_guess'] is not None and obs['species_guess'] != "None": # 'None'文字列も除外
        observation_to_species[str(obs['id'])] = obs['species_guess']

images = []
labels = []

# ダウンロードした画像を読み込み、前処理とラベル付けを行う
print("画像の読み込み、前処理、ラベル付けを開始します...")
for filename in os.listdir(DOWNLOAD_DIR):
    if filename.endswith(".png"):
        obs_id = filename.split('_')[1].split('.')[0]
        if obs_id in observation_to_species: # species_guessがある画像のみを対象
            img_path = os.path.join(DOWNLOAD_DIR, filename)
            try:
                img = Image.open(img_path).convert('RGB') # RGBに変換して3チャンネルにする
                img = img.resize(IMAGE_SIZE)             # 指定サイズにリサイズ
                img_array = np.array(img) / 255.0         # ピクセル値を0-1に正規化
                images.append(img_array)
                labels.append(observation_to_species[obs_id])
            except Exception as e:
                print(f"画像 {filename} の処理中にエラーが発生しました: {e}")

# リストをNumPy配列に変換
X = np.array(images)
y_species = np.array(labels)

# ラベルを数値にエンコード (最初のエンコード)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_species)

# クラスごとのサンプル数をカウント
unique_classes, counts = np.unique(y, return_counts=True)

# サンプル数が2つ未満のクラスを特定
single_sample_classes = unique_classes[counts < 2]

# サンプル数が2つ以上のクラスに属するデータのみをフィルタリング
filtered_indices = [i for i, label_val in enumerate(y) if label_val not in single_sample_classes]

# Initialize variables to avoid NameError if filtered_indices is empty
X_filtered = np.array([])
y_filtered = np.array([])
new_label_encoder = None
num_classes = 0 # Initialize num_classes
X_train, X_test, y_train, y_test = np.array([]), np.array([]), np.array([]), np.array([])

if len(filtered_indices) == 0:
    print("データセットに訓練/テスト分割に必要な十分なサンプルを持つクラスがありません。")
    # X_train, X_test, y_train, y_test are already initialized to empty arrays
    # X_filtered, y_filtered, num_classes are also initialized above
else:
    X_filtered = X[filtered_indices]
    y_filtered_raw_labels = y[filtered_indices]

    # ここで新しいLabelEncoderを作成し、y_filtered_raw_labelsを再エンコードして連続したラベルにする
    # これにより、Kerasが期待する0からの連続したクラスインデックスが保証されます。
    new_label_encoder = LabelEncoder()
    y_filtered = new_label_encoder.fit_transform(y_filtered_raw_labels)
    num_classes = len(np.unique(y_filtered)) # Define num_classes here

    # データセットを訓練用とテスト用に分割（フィルタリング後にstratifyを使用）
    X_train, X_test, y_train, y_test = train_test_split(X_filtered, y_filtered, test_size=0.5, random_state=42, stratify=y_filtered)


print("\nデータセットの準備が完了しました。")
print(f"総画像数: {len(X_filtered)}")
# Handle empty array shape for printing
print(f"画像の形状: {X_filtered.shape[1:] if X_filtered.shape else IMAGE_SIZE + (3,)}")
print(f"クラス数: {num_classes}")
print(f"訓練データ数: {len(X_train)}")
print(f"テストデータ数: {len(X_test)}")

# ラベルエンコーダが学習したクラス名を表示
if new_label_encoder is not None: # new_label_encoderが定義されている場合のみ表示
    print("\n検出された昆虫の種別:")
    for i, class_name in enumerate(new_label_encoder.classes_):
        print(f"  {i}: {class_name}")

画像の読み込み、前処理、ラベル付けを開始します...

データセットの準備が完了しました。
総画像数: 1644
画像の形状: (128, 128, 3)
クラス数: 212
訓練データ数: 822
テストデータ数: 822

検出された昆虫の種別:
  0: 3
  1: 4
  2: 5
  3: 7
  4: 8
  5: 10
  6: 11
  7: 12
  8: 13
  9: 18
  10: 19
  11: 20
  12: 25
  13: 27
  14: 30
  15: 31
  16: 33
  17: 35
  18: 36
  19: 40
  20: 41
  21: 43
  22: 44
  23: 45
  24: 46
  25: 47
  26: 49
  27: 50
  28: 53
  29: 54
  30: 55
  31: 56
  32: 62
  33: 64
  34: 70
  35: 71
  36: 73
  37: 77
  38: 78
  39: 79
  40: 80
  41: 81
  42: 83
  43: 84
  44: 85
  45: 89
  46: 97
  47: 99
  48: 106
  49: 109
  50: 114
  51: 117
  52: 119
  53: 120
  54: 121
  55: 122
  56: 124
  57: 125
  58: 130
  59: 131
  60: 133
  61: 135
  62: 136
  63: 138
  64: 139
  65: 141
  66: 142
  67: 145
  68: 146
  69: 150
  70: 153
  71: 154
  72: 156
  73: 157
  74: 158
  75: 163
  76: 166
  77: 172
  78: 174
  79: 177
  80: 180
  81: 182
  82: 184
  83: 187
  84: 188
  85: 189
  86: 191
  87: 192
  88: 193
  89: 194
  90: 196
  91: 199
  92: 201
  93: 2

## 3. 機械学習モデルの構築と訓練

ここでは、シンプルな畳み込みニューラルネットワーク（CNN）を構築し、訓練データを使用してモデルを訓練します。最終的な目標であるラズパイでの使用を考慮し、軽量なモデルを設計することが望ましいですが、まずは基本的な構成で進めます。

### データ拡張の導入

データセットが限られているため、**データ拡張 (Data Augmentation)** を適用して、モデルがより多様なデータで学習できるようにします。これにより、過学習を防ぎ、モデルの汎化性能を高めることができます。

In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# データ拡張の設定
# ここでは、ランダムな回転、幅/高さシフト、せん断、ズーム、水平反転を適用します。
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest' # 変換で空白ができた場合の埋め方
)

# 訓練データにデータ拡張を適用するためのジェネレータを作成
train_generator = datagen.flow(
    X_train, y_train,
    batch_size=len(X_train) # バッチサイズを訓練データの全数に設定（小規模データセット向け）
)

print("データ拡張ジェネレータが設定されました。")
print(f"訓練データ拡張後のバッチサイズ: {train_generator.batch_size}")
# print(f"訓練データ拡張後のサンプル数 (1エポックあたり): {train_generator.samples}") # この行がエラーの原因だったため削除

# テストデータは拡張しないため、直接使用します。


データ拡張ジェネレータが設定されました。
訓練データ拡張後のバッチサイズ: 822


上記のデータ拡張ジェネレータを使用してモデルを訓練します。これにより、モデルは各エポックで新しい変換された画像を学習し、より堅牢な特徴を学習できるようになります。

In [11]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# クラス数の定義
num_classes = len(np.unique(y_filtered)) # y_filtered は前処理済みのラベル

# データセットが訓練可能か確認
if num_classes == 0 or len(X_train) == 0 or len(y_train) == 0:
    print("訓練データが不足しているため、モデルを訓練できません。より多くのデータを取得するか、クラス数を調整してください。")
else:
    print(f"モデルを訓練中。クラス数: {num_classes}、訓練データ数: {len(X_train)}")

    # シンプルなCNNモデルの構築
    model = keras.Sequential([
        keras.Input(shape=IMAGE_SIZE + (3,)), # 画像サイズ (128, 128, 3)
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax') # 多クラス分類の場合softmax
    ])

    # モデルのコンパイル
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy', # ラベルが整数のためsparse_categorical_crossentropy
        metrics=['accuracy']
    )

    # モデルの概要を表示
    model.summary()

    # モデルの訓練
    # 訓練データにデータ拡張を適用したジェネレータを使用
    history = model.fit(
        train_generator, # データ拡張された訓練データを使用
        epochs=30, # 訓練データが少ないため、エポック数を増やして訓練を強化
        validation_data=(X_test, y_test) # テストデータで検証
    )

    print("\nモデルの訓練が完了しました。")

モデルを訓練中。クラス数: 212、訓練データ数: 822


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,605,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 212)            │        13,780 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,712,724 (6.53 MB)

 Trainable params: 1,712,724 (6.53 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 44s 44s/step - accuracy: 0.0097 - loss: 5.3412 - val_accuracy: 0.0438 - val_loss: 5.3028
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 38s 38s/step - accuracy: 0.0450 - loss: 5.2921 - val_accuracy: 0.0608 - val_loss: 5.1887
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 39s 39s/step - accuracy: 0.0693 - loss: 5.1974 - val_accuracy: 0.0620 - val_loss: 5.1712
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 43s 43s/step - accuracy: 0.0645 - loss: 5.1821 - val_accuracy: 0.0633 - val_loss: 5.1377
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 36s 36s/step - accuracy: 0.0633 - loss: 5.1444 - val_accuracy: 0.0633 - val_loss: 5.1039
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 37s 37s/step - accuracy: 0.0633 - loss: 5.1053 - val_accuracy: 0.0633 - val_loss: 5.0858
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 38s 38s/step - accuracy: 0.0633 - loss: 5.0787 - val_accuracy: 0.0633 - val_loss: 5.0486
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 39s 39s/step - accuracy: 0.0633 - loss: 5.0447 - val_accuracy: 0.0572 - val_loss: 5.0184


## 4. モデルのTensorFlow Lite形式への変換

訓練したモデルをRaspberry Piのようなエッジデバイスで利用するためには、TensorFlow Lite形式に変換することが推奨されます。これにより、モデルのサイズが小さくなり、推論速度が向上します。

In [12]:
import tensorflow as tf

# モデルが訓練されているか確認
if 'model' in locals():
    # KerasモデルをTensorFlow Liteコンバータで変換
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # 最適化オプションを設定（デフォルトではサイズ最適化が適用されます）
    # 更なる最適化として、量子化を適用することも可能ですが、まずはデフォルトで変換します。
    # converter.optimizations = [tf.lite.Optimize.DEFAULT]

    tflite_model = converter.convert()

    # 変換したモデルをファイルに保存
    tflite_model_path = 'insect_classifier.tflite'
    with open(tflite_model_path, 'wb') as f:
        f.write(tflite_model)

    print(f"TensorFlow Liteモデルが '{tflite_model_path}' に保存されました。")
else:
    print("モデルが見つかりません。まずモデルを訓練してください。")

Saved artifact at '/tmp/tmpr77s98r5'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 212), dtype=tf.float32, name=None)
Captures:
  133156072011920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133156072013072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133156072010960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133156072013264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133156072012304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133156072013456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133156072010384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133156072012496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133156072011728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133156072010576: TensorSpec(shape=(), dtype=tf.resource, name=None)
TensorFlow Lit

In [13]:
import os

# モデルが保存されたパス
model_path = 'insect_classifier.tflite'

# 現在のディレクトリにファイルが存在するか確認
if os.path.exists(model_path):
    print(f"ファイル '{model_path}' が現在のディレクトリに存在します。")
    print(f"絶対パス: {os.path.abspath(model_path)}")
else:
    print(f"ファイル '{model_path}' が見つかりません。")

# また、ファイル一覧を表示して確認することもできます。
# !ls -F

ファイル 'insect_classifier.tflite' が現在のディレクトリに存在します。
絶対パス: /content/insect_classifier.tflite
